# Proj2 — Light Full-Data 2-Config Sweep

This notebook is a low-memory version of the project sweep for DSMLP MIG.

**What it does:**
- Uses full `train.json` + `validation.json`
- Runs only **2 configs** (0.5B narrow and 1.5B narrow)
- Enables **QLoRA** to reduce VRAM pressure
- Uses conservative training args (`batch=1`, `max_length=512`, `num_chunks=1`)

| Catalog ID | Model | LR | LoRA |
|------------|-------|-----|------|
| 0 | Qwen2.5-0.5B-Instruct | 5e-5 | narrow (r=8) |
| 1 | Qwen2.5-1.5B-Instruct | 5e-5 | narrow (r=8) |

RapidFire assigns internal run ids `1..N` for selected configs.

## 0. Debug knobs (edit here)

In [ ]:
from datetime import datetime

from paths import LOGS_DIR, REPO_ROOT, TRAIN_JSON, VAL_JSON, print_paths, require_release_data, setup

setup(chdir_try=True)  # cwd → try/; ./logs → try/logs; imports schema_*.py from here
require_release_data()
print_paths()

# --- What to run (catalog ids) ---
RUN_IDS = [0, 1]    # 0: 0.5B narrow, 1: 1.5B narrow

# --- Full-data but light training ---
NUM_EPOCHS = 1
MAX_LENGTH = 2048 # increase this for less truncation after step 2
USE_QLORA = False # set this to reduce issues 

# --- Experiment naming (timestamp avoids RapidFire suffix churn) ---
EXPERIMENT_BASE = "proj2-debug-2config"
EXPERIMENT_NAME = f"{EXPERIMENT_BASE}-{datetime.now():%Y%m%d-%H%M}"
EXPERIMENT_PATH = "./logs"

# --- RapidFire scheduling ---
NUM_CHUNKS = 1      # reduce allocator fragmentation (worker reset per chunk)
NUM_GPUS = 1
SEED = 42

SCHEMA_STYLE = "full"         # or "full"

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"RUN_IDS={RUN_IDS}  epochs={NUM_EPOCHS}  max_length={MAX_LENGTH}  qlora={USE_QLORA}")
print(f"NUM_CHUNKS={NUM_CHUNKS}  (full train + val)")

## 1. Optional — free GPU / end stale experiment

In [ ]:
import gc

def cuda_cleanup():
  """Call between failed runs to reduce allocator/NVML pressure."""
  gc.collect()
  try:
    import torch
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
      torch.cuda.synchronize()
      print("CUDA cache cleared")
  except Exception as e:
    print(f"cuda_cleanup skipped: {e}")

# Uncomment if a previous Experiment object is still in memory:
# experiment.end()
# cuda_cleanup()

## 2. Imports & data

In [ ]:
import json
from functools import lru_cache

from datasets import Dataset
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

from schema_formatting import make_formatting_func
from schema_metrics import make_compute_metrics

# TRAIN_JSON / VAL_JSON set in section 0 via paths.setup()
with open(TRAIN_JSON) as f:
    raw_train = json.load(f)
with open(VAL_JSON) as f:
    raw_val = json.load(f)

train_ds = Dataset.from_list(raw_train)
val_ds = Dataset.from_list(raw_val)
train_ds = train_ds.shuffle(seed=SEED)
fmt_func = make_formatting_func(SCHEMA_STYLE)

print(f"Train={len(train_ds)}  Val={len(val_ds)}  schema={SCHEMA_STYLE}")

## 3. Model factory & config grid (2 lightweight runs)

In [ ]:
@lru_cache
def get_tokenizer(model_name: str):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def create_model(model_config: dict):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_kwargs = model_config["model_kwargs"].copy()
    use_qlora = model_kwargs.pop("use_qlora", False)

    if use_qlora:
        from peft import prepare_model_for_kbit_training
        from transformers import BitsAndBytesConfig
        import torch
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_cfg, **model_kwargs
        )
        model.enable_input_require_grads()   # required for QLoRA + gradient_checkpointing
        
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id
    return (model, tokenizer)


peft_narrow = RFLoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
def make_sft_args(lr: float) -> RFSFTConfig:
    return RFSFTConfig(
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05, # added warm up loss keeps exploding!
        max_grad_norm=0.3, # cap the grad
        weight_decay=0.01, # prevent from memorizing
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=NUM_EPOCHS,
        max_length=MAX_LENGTH,
        logging_steps=10,
        eval_strategy="epoch",
        load_best_model_at_end=False,
        gradient_checkpointing=True,
        # gradient_checkpointing=False, # # caused an error for QLoRA
        bf16=True,
        fp16=False,
        report_to="none",
    )


def model_kwargs() -> dict:
    kw = {"device_map": "auto", "torch_dtype": "auto", "use_cache": False}
    if USE_QLORA:
        kw["use_qlora"] = True
    return kw


GEN_CFG = {"max_new_tokens": 128, "do_sample": False, "temperature": 1.0}

QWEN_05B = "Qwen/Qwen2.5-0.5B-Instruct"
QWEN_15B = "Qwen/Qwen2.5-1.5B-Instruct"

RUN_CATALOG = [
    {"run_id": 0, "model": QWEN_05B, "lr": 5e-5, "label": "0.5B | lr=5e-5 | narrow"}, # make lr smaller
    {"run_id": 1, "model": QWEN_15B, "lr": 5e-5, "label": "1.5B | lr=5e-5 | narrow"},
]


def catalog_entry_to_rf_config(entry: dict) -> RFModelConfig:
    return RFModelConfig(
        model_name=entry["model"],
        peft_config=peft_narrow,
        training_args=make_sft_args(entry["lr"]),
        model_type="causal_lm",
        model_kwargs=model_kwargs(),
        formatting_func=fmt_func,
        # compute_metrics=compute_metrics_for_rf, # remote this -> causing the 0.0149 score
        compute_metrics=make_compute_metrics(get_tokenizer(entry["model"])),
        generation_config=GEN_CFG.copy(),
    )


selected = [e for e in RUN_CATALOG if e["run_id"] in RUN_IDS]
if not selected:
    raise ValueError(f"RUN_IDS {RUN_IDS} matched no runs")

for i, e in enumerate(selected, start=1):
    print(f"  catalog {e['run_id']} -> run ~{i}: {e['label']}  qlora={USE_QLORA}")

config_set = List([catalog_entry_to_rf_config(e) for e in selected])
config_group = RFGridSearch(configs=config_set, trainer_type="SFT")

print(f"\nLaunching {len(selected)} run(s) on {len(train_ds)} train / {len(val_ds)} val examples")

## 4. Create experiment & run

In [ ]:
cuda_cleanup()

experiment = Experiment(
    experiment_name=EXPERIMENT_NAME,
    mode="fit",
    experiment_path=EXPERIMENT_PATH,
)
print(f"Logs: {EXPERIMENT_PATH}/{experiment.experiment_name}")

In [ ]:
cfg = catalog_entry_to_rf_config(selected[0])
print(cfg.training_args.learning_rate)
print(cfg.training_args.warmup_ratio)
print(cfg.training_args.max_grad_norm)

In [7]:
# Full-data light run (2 configs). If OOM persists: lower MAX_LENGTH or NUM_EPOCHS in section 0.
experiment.run_fit(
    config_group,
    create_model,
    train_ds,
    val_ds,
    num_chunks=NUM_CHUNKS,
    seed=SEED,
    num_gpus=NUM_GPUS,
)

KeyboardInterrupt: 

## 5. Results & cleanup

In [ ]:
results = experiment.get_results()
display(results)

In [8]:
experiment.end()
cuda_cleanup()
print("Experiment ended.")

Task Training and Validation cancelled
Experiment proj2-debug-2config-20260528-2306 ended
Workers stopped
CUDA cache cleared
Experiment ended.
